<a href="https://colab.research.google.com/github/Riyen-Shah/Veritas-AI-International-Highschool-Hackathon-2026/blob/main/Unreal_Madrid_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Credible: a bank loan predictor with reduced bias for improved profit**
# **Veritas AI International High School Hackathon 2026 · TEAM UNREAL MADRID SUBMISSION**

Track: AI + Finance (built on suggested project 2.3, Fairness audit of a loan model) Team: Unreal Madrid; Rayaan Vakas and Riyen Shah

Credible helps banks make fair and cost effective decisions when deciding whether loan applicants are good or bad risks, while minimising bias and losses.

How to run: Runtime → Run all. It takes under a minute and needs no API keys or uploads.

**Disclaimer:**
***This project is a prototype and is not financial advice***



---

# **1. Install, Import and  Load the data**

We use the *Statlog (German Credit Data)* dataset (Hans Hofmann, UC Irvine Machine Learning Repository): 1,000 real loan applications, each containing 20 columns of information about the applicant, and whether they were accepted.

In [ ]:
!pip install ucimlrepo
!pip install -U scikit-learn
!pip install numpy
!pip install pandas

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# fetch dataset
statlog_german_credit_data = fetch_ucirepo(id=144)
# data (as pandas dataframes)
X = statlog_german_credit_data.data.features
y = statlog_german_credit_data.data.targets
# metadata
print(statlog_german_credit_data.metadata)
# variable information
print(statlog_german_credit_data.variables)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.0 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
{'uci_id': 144, 'name': 'Statlog (German Credit Data)', 'repository_url': 'https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data', 'data_url': 'https://archive.ics.uci.edu/static/public/144/data.csv', 'abstract': 'This dataset classifies people described by a set of attributes as good or bad credit risks. Comes in two formats (one all numeric). Also comes with a cost matrix', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1000, 'num_features': 20, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Other', 'Marital Status', 'Age', 'Occupation'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_

---
#**2. CLEAN AND ENCODE THE DATA**
We cleaned and re-encoded the data to make it easily accessible by our artificial intelligence model, and printed the amount of people who were deemed "Good" for a loan, and those deemed "Bad" for a loan.

In [ ]:
# 1. Clean the target (y):
# The dataset has classes: 1 = Good credit risk, 2 = Bad credit risk.
# Let's map it so 1 = Good (1) and 2 = Bad (0).
y_binary = y['class'].map({1: 1, 2: 0})

# 2. Encode all categorical features in X:
# pd.get_dummies automatically detects text columns (like A11, A12)
# and converts them into separate 0/1 binary indicator columns.
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Original X shape: {X.shape}")
print(f"Encoded X shape (after turning text into numbers): {X_encoded.shape}")
print(f"Target distribution (1 = Good, 0 = Bad):\n{y_binary.value_counts()}")

Original X shape: (1000, 20)
Encoded X shape (after turning text into numbers): (1000, 48)
Target distribution (1 = Good, 0 = Bad):
class
1    700
0    300
Name: count, dtype: int64


---
#**3. Train/Test split**
We split our data so the model trains using a randomly generated 80% of it, and is tested on the remaining 20%.

In [ ]:
# 1. Split data into training (80%) and testing (20%) sets
# stratify=y_binary ensures both sets have the same ratio of good/bad credit risks
X_train, X_test, y_train, y_test = train_test_split(
X_encoded, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print(f"Training set: {len(X_train)} applicants")
print(f"Test set: {len(X_test)} applicants\n")

Training set: 800 applicants
Test set: 200 applicants



---
#**4. Test model 1**
We tested model 1 and compared it against our baseline and saw an improvement, however we knew the model could be more accurate.

---
#**5. Reveal biases and define "problem groups"**
We discovered people aged 30 or under, or a loan of 13-24 months were being accepted more frequently, including people who should not have been accepted.

In [ ]:
# 2. Mark the problem groups: aged 30 or under, OR a loan of 13-24 months
problem_train = ((X_train['Attribute13'] <= 30) | X_train['Attribute2'].between(13, 24)).values
problem_test = ((X_test['Attribute13'] <= 30) | X_test['Attribute2'].between(13, 24)).values

---
#**6. Fix the biases by weighting applicants in the problem groups**
We countered the bias by using a weight coefficient to give more weight to the bad risk applicants in the "problem groups". Essentially, we multiplied the number of bad applicants in the groups by a constant to improve how seriously our model treats them in the test data, hence countering bias.

In [ ]:
# 3. Train the model, giving bad-risk applicants in the problem groups 3x weight
K = 3
weights = np.where(problem_train & (y_train == 0).values, K, 1)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train, sample_weight=weights)

# 4. Make predictions on the test set
y_pred = model.predict(X_test)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


---
#**7. Evaluate the performance of the new weighted model**
We evaluated the performance of the new weighted model, and found out its accuracy was worse than the origional, but fine tuned its weighted coefficient to prioritise decreasing the number of bad risk applicants being accepted despite accuracy being decreased, to prevent money loss.

In [ ]:
# 5. Evaluate performance
bad_approved = y_pred[problem_test & (y_test == 0).values].mean()        # bad risks wrongly approved
good_rejected = 1 - y_pred[problem_test & (y_test == 1).values].mean()   # good risks wrongly rejected

print(f"--- Model Performance (bad-risk weight k={K}) ---")
print(f"Accuracy:      {accuracy_score(y_test, y_pred):.1%}")
print(f"Precision:     {precision_score(y_test, y_pred):.1%} (of those predicted 'good', how many were actually good)")
print(f"Recall:        {recall_score(y_test, y_pred):.1%} (of actual 'good' credit risks, how many did we catch)")
print(f"Bad approved:  {bad_approved:.1%} (of bad risks aged 30 or under or on 13-24 month loans, how many were wrongly approved)")
print(f"Good rejected: {good_rejected:.1%} (of good risks in those same groups, how many were wrongly rejected)")

print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

--- Model Performance (bad-risk weight k=3) ---
Accuracy:      68.0%
Precision:     83.9% (of those predicted 'good', how many were actually good)
Recall:        67.1% (of actual 'good' credit risks, how many did we catch)
Bad approved:  33.3% (of bad risks aged 30 or under or on 13-24 month loans, how many were wrongly approved)
Good rejected: 36.6% (of good risks in those same groups, how many were wrongly rejected)

Detailed Report:
              precision    recall  f1-score   support

           0       0.48      0.70      0.57        60
           1       0.84      0.67      0.75       140

    accuracy                           0.68       200
   macro avg       0.66      0.69      0.66       200
weighted avg       0.73      0.68      0.69       200



---
#**8. Bank cost**
We know the cost to the bank is 5 times worse when accepting a bad risk loan than rejecting a good risk loan, so we calculated how much money this model would cost the bank, and compared it to the baseline and model 1.
We could see a drastic decrease in cost from this model as it prioritised rejecting bad risk loans, saving the bank more money.

In [ ]:
#bank cost rating

bank_cost = 5 * ((y_pred == 1) & (y_test == 0).values).sum() + ((y_pred == 0) & (y_test == 1).values).sum()
print(f"Bank cost:     {bank_cost} (5 x bad risks approved + 1 x good risks rejected; lower is better)")

Bank cost:     136 (5 x bad risks approved + 1 x good risks rejected; lower is better)


#**Credits**


*   https://claude.ai debugging code to implement weight coeffeicient
*   https://gemini.google.com/ debugging code to pip install libraries and train/test code  debugging.


*   https://chatgpt.com/ - debugging
*   https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data - dataset



